# 06 · Answer with citations

Stage 7 of the pipeline (see `ARCHITECTURE.md`): turn the best 5 chunks into a written answer.

1. **Retrieve + rerank** (same as `05`): top 20 from Chroma, best 5 by the cross-encoder.
2. **Answer**: a free, local LLM (`phi4-mini`, running in **Ollama**) reads the question and those 5 chunks and writes a short answer, citing chunk ids like `[fixed-0063]`. No API key, nothing leaves the laptop.
3. **Verify**, in plain code, no second LLM: is every citation one of the 5 chunks, and does every number in the answer actually appear in them? Small models sometimes invent numbers; this catches it.

Before running:
- Ollama installed and running, and `ollama pull phi4-mini` done.
- `pip install langchain-ollama` in the `.venv`.

⚠️ Memory: ~850 MB in this notebook (all-MiniLM + reranker) **plus ~3–3.5 GB in Ollama** for phi4-mini, so ~4–4.5 GB in total. Close the browser and other heavy apps, and restart other notebook kernels first. Each answer takes roughly 20–60 seconds on CPU.

Reads `data/chroma_db/`. Writes nothing.

In [1]:
import os
import re
import time
from pathlib import Path

# Hugging Face warns that Windows can't use symlinks in its model cache; caching still works.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from sentence_transformers import CrossEncoder

CHROMA_DIR = Path("../data/chroma_db")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "phi4-mini"
STRATEGY = "fixed"  # looked slightly more reliable in 05; try "semantic" too
RETRIEVE_K = 20
KEEP = 5

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
store = Chroma(collection_name=f"acre_{STRATEGY}", embedding_function=embeddings, persist_directory=str(CHROMA_DIR))
reranker = CrossEncoder(RERANK_MODEL)

# temperature=0: always pick the most likely next word, so the same question gives the same answer.
# num_ctx: how many tokens the model reads at once; 5 chunks + instructions fit well under 4096.
llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=4096)

print(f"acre_{STRATEGY}: {store._collection.count()} vectors · reranker and {LLM_MODEL} ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

acre_fixed: 324 vectors · reranker and phi4-mini ready


Check that Ollama answers before going further. If this fails, start the Ollama app (or run `ollama serve` in a terminal) and check `ollama list` shows `phi4-mini`. The first call also loads the model into RAM, so it's slower.

In [2]:
start = time.perf_counter()
reply = llm.invoke("Reply with just the word: ready")
print(reply.content, f"({time.perf_counter() - start:.1f} s)")

ready (4.4 s)


## The three steps

- `retrieve()`: the `05` search, keeping only the best 5.
- `answer()`: builds the prompt (rules + the 5 chunks, each labelled with its id) and asks the LLM.
- `verify()`: checks citations and numbers against the 5 chunks.

The rules in `INSTRUCTIONS` carry what we learned in `05`: "this quarter" means the three months ended June 30, 2026, not the six months, and ACRE reports net interest margin in dollars.

In [3]:
INSTRUCTIONS = """You answer questions about the Form 10-Q of Ares Commercial Real Estate Corporation (ACRE) \
for the quarter ended June 30, 2026, using ONLY the excerpts provided.

Rules:
- "This quarter" means the three months ended June 30, 2026, not the six months.
- After every fact, cite the excerpt it came from with its id in square brackets, e.g. [fixed-0063].
- Copy numbers exactly as written in the excerpts. Do not calculate or round.
- If the excerpts do not contain the answer, reply exactly: Not found in the provided excerpts.
- Answer in 2 to 4 sentences."""

NOT_FOUND = "Not found in the provided excerpts."


def retrieve(question):
    hits = store.similarity_search(question, k=RETRIEVE_K)
    scores = reranker.predict([(question, doc.page_content) for doc in hits])
    ranked = sorted(zip(scores, hits), key=lambda pair: pair[0], reverse=True)
    return [doc for _, doc in ranked[:KEEP]]


def answer(question, chunks, instructions=INSTRUCTIONS):
    excerpts = "\n\n".join(
        f"[{c.metadata['chunk_id']}] ({c.metadata['section']})\n{c.page_content}" for c in chunks
    )
    messages = [
        ("system", instructions),
        ("human", f"Excerpts:\n\n{excerpts}\n\nQuestion: {question}"),
    ]
    return llm.invoke(messages).content.strip()


def numbers_in(text):
    """Numbers like 287.3, 0.15 or 1,234 (commas removed), skipping chunk ids inside [...]."""
    text = re.sub(r"\[[^\]]*\]", " ", text)
    return {n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}


def verify(text, chunks):
    ids = {c.metadata["chunk_id"] for c in chunks}
    cited = set(re.findall(r"\[((?:fixed|semantic)-\d{4})\]", text))
    source_numbers = numbers_in("\n".join(c.page_content for c in chunks))
    return {
        "not_found": text.startswith(NOT_FOUND),
        "has_citation": bool(cited),
        "unknown_citations": sorted(cited - ids),
        "numbers_not_in_excerpts": sorted(numbers_in(text) - source_numbers),
    }


def ask(question):
    start = time.perf_counter()
    chunks = retrieve(question)
    text = answer(question, chunks)
    checks = verify(text, chunks)
    ok = checks["not_found"] or (
        checks["has_citation"] and not checks["unknown_citations"] and not checks["numbers_not_in_excerpts"]
    )

    print(f"Q: {question}\n")
    print(text, "\n")
    print("Sources given to the model:", ", ".join(c.metadata["chunk_id"] for c in chunks))
    print("Checks:", "PASS" if ok else "CHECK THIS ANSWER", checks)
    print(f"({time.perf_counter() - start:.0f} s)")
    return text, chunks, checks

## One question

The question from `04` and `05`. The right answer is approximately **$8.6 million** for the three months ended June 30, 2026 (vs. $7.0 million a year earlier), from `fixed-0180`. Watch whether the model picks the three-month figure over the six-month one ($16.1 million).

In [4]:
text, chunks, checks = ask("What was the net interest margin this quarter?")

Q: What was the net interest margin this quarter?

Not found in the provided excerpts. 

Sources given to the model: fixed-0182, fixed-0180, fixed-0047, fixed-0280, fixed-0310
Checks: PASS {'not_found': True, 'has_citation': False, 'unknown_citations': [], 'numbers_not_in_excerpts': []}
(13 s)


## More questions

The same questions as `05`. Known answers so far, from reading the chunks:

- Non-accrual: **three loans, Carrying Value $287.3 million** (fixed-0063); one is the California subordinated C-Note (fixed-0060).
- Dividend: fixed-0161 is the **$0.15** dividend for the **third** quarter of 2026, declared after June 30 (a subsequent event). The second-quarter dividend isn't in that chunk, so "Not found" is a fair answer if no other excerpt has it.
- Loans held for investment: **$1,748,835** in the table (fixed-0281). The table is in thousands, so that's about **$1.75 billion**.
- CECL reserve: **$139.1 million** for loans held for investment (fixed-0071).

Read each answer and its cited chunks to judge. ~10–60 s each.

In [5]:
QUESTIONS = [
    "Which loans are on non-accrual status?",
    "What dividend per share was declared for the quarter?",
    "What was the total carrying value of loans held for investment at June 30, 2026?",
    "How large is the current expected credit loss (CECL) reserve?",
    "Who is the company's auditor?",  # may well not be in a 10-Q: tests the "Not found" rule
]

for q in QUESTIONS:
    ask(q)
    print("\n" + "─" * 100 + "\n")

Q: Which loans are on non-accrual status?

As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $287.3 million. As of December 31, 2025, the Company had four loans held for investment on non-accrual status with a Carrying Value of $308.1 million. Additionally, as of June 30, 2026, the subordinated C-Note was on non-accrual status. Not found in the provided excerpts. 

Sources given to the model: fixed-0031, fixed-0063, fixed-0060, fixed-0075, fixed-0044
Checks: CHECK THIS ANSWER {'not_found': False, 'has_citation': False, 'unknown_citations': [], 'numbers_not_in_excerpts': []}
(16 s)

────────────────────────────────────────────────────────────────────────────────────────────────────

Q: What dividend per share was declared for the quarter?

Not found in the provided excerpts. 

Sources given to the model: fixed-0161, fixed-0178, fixed-0306, fixed-0271, fixed-0274
Checks: PASS {'not_found': True, 'has_citation': False, 'unk

## Option A: quote first

The first prompt gave a small model five rules at once, and "Not found" became an easy way out. Even when the answer was in the excerpts (net interest margin, `fixed-0180`), the model sometimes took it.

The new prompt (`INSTRUCTIONS_V2`) makes the model **find the evidence before it answers**, in a fixed format:

```
QUOTE:  the sentence from the excerpts that answers the question, copied word for word
SOURCE: [the id of that excerpt]
ANSWER: one or two sentences, with numbers exactly as in the quote
```

- **Quote first.** Copying a sentence is easier for a small model than judging whether an answer exists, and once the right sentence is written down the answer usually follows.
- **"Not found" only if no excerpt is about the question at all**, so it's a last resort, not an escape.
- **A fixed format is easy to check in code.** `verify_v2()` confirms the quote really appears in the cited chunk, word for word. This catches made-up quotes as well as made-up numbers.

In [6]:
INSTRUCTIONS_V2 = """You answer questions about the Form 10-Q of Ares Commercial Real Estate Corporation (ACRE) \
for the quarter ended June 30, 2026. Use only the excerpts provided. "This quarter" means the three months \
ended June 30, 2026.

Reply in exactly this format:
QUOTE: <the sentence or table row from the excerpts that answers the question, copied word for word>
SOURCE: <the id of that excerpt, e.g. [fixed-0063]>
ANSWER: <one or two sentences answering the question, with numbers exactly as they appear in the quote>

Only if no excerpt is about the question at all, reply:
QUOTE: none
SOURCE: none
ANSWER: Not found in the provided excerpts."""

FIELDS = r"(QUOTE|SOURCE|ANSWER):\s*(.*?)(?=\n\s*(?:QUOTE|SOURCE|ANSWER):|\Z)"


def parse(text):
    """Split a reply into its QUOTE / SOURCE / ANSWER parts (missing parts come back as "")."""
    found = dict(re.findall(FIELDS, text, flags=re.S))
    return {key: found.get(key, "").strip() for key in ("QUOTE", "SOURCE", "ANSWER")}


def normalize(text):
    """Lower-case, no spaces or quote marks: so a quote still matches if spacing or ’ vs ' differ."""
    return re.sub(r"[\s\"'“”‘’]+", "", text).lower().rstrip(".…")


def verify_v2(parts, chunks):
    by_id = {c.metadata["chunk_id"]: c.page_content for c in chunks}
    cited = re.findall(r"(?:fixed|semantic)-\d{4}", parts["SOURCE"])
    cited_text = "\n".join(by_id.get(i, "") for i in cited)
    return {
        "format_ok": all(parts.values()),
        "not_found": parts["ANSWER"].startswith(NOT_FOUND),
        "unknown_source": sorted(set(cited) - set(by_id)),
        "quote_in_source": bool(cited_text) and normalize(parts["QUOTE"]) in normalize(cited_text),
        "numbers_not_in_excerpts": sorted(numbers_in(parts["ANSWER"]) - numbers_in("\n".join(by_id.values()))),
    }


def passed_v2(checks):
    if checks["not_found"]:
        return True
    return (checks["format_ok"] and checks["quote_in_source"]
            and not checks["unknown_source"] and not checks["numbers_not_in_excerpts"])

### Old vs. new prompt, same chunks

Each question is retrieved **once**, and both prompts get exactly the same 5 chunks. Only the prompt changes, so any difference in the answers comes from the prompt alone.

Expected: net interest margin **$8.6 million** (three months); non-accrual **three loans, $287.3 million**; dividend for Q2 **not in these chunks** (the $0.15 is for Q3); loans held for investment **$1,748,835 thousand**; CECL reserve **$139.1 million**; auditor **not found**.

12 model calls, so about 3–4 minutes.

In [7]:
import pandas as pd

COMPARE = [
    "What was the net interest margin this quarter?",
    "Which loans are on non-accrual status?",
    "What dividend per share was declared for the quarter?",
    "What was the total carrying value of loans held for investment at June 30, 2026?",
    "How large is the current expected credit loss (CECL) reserve?",
    "Who is the company's auditor?",
]


def problems_v2(checks):
    if checks["not_found"]:
        return "said not found"
    issues = {
        "format": not checks["format_ok"],
        "quote not in source": not checks["quote_in_source"],
        f"unknown source {checks['unknown_source']}": bool(checks["unknown_source"]),
        f"numbers not in excerpts {checks['numbers_not_in_excerpts']}": bool(checks["numbers_not_in_excerpts"]),
    }
    return ", ".join(name for name, bad in issues.items() if bad) or "none"


rows = []
for q in COMPARE:
    chunks = retrieve(q)

    old = answer(q, chunks)
    old_checks = verify(old, chunks)
    old_ok = old_checks["not_found"] or (old_checks["has_citation"] and not old_checks["unknown_citations"]
                                         and not old_checks["numbers_not_in_excerpts"])

    new = answer(q, chunks, instructions=INSTRUCTIONS_V2)
    parts = parse(new)
    new_checks = verify_v2(parts, chunks)

    rows.append({
        "question": q,
        "old answer": old,
        "old pass": old_ok,
        "new answer": parts["ANSWER"] or new,  # whole reply if the model ignored the format
        "new source": parts["SOURCE"],
        "new pass": passed_v2(new_checks),
        "new problems": problems_v2(new_checks),
        "new quote": parts["QUOTE"],
    })
    print(f"done: {q}")

pd.set_option("display.max_colwidth", None)
comparison = pd.DataFrame(rows)
comparison.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

done: What was the net interest margin this quarter?
done: Which loans are on non-accrual status?
done: What dividend per share was declared for the quarter?
done: What was the total carrying value of loans held for investment at June 30, 2026?
done: How large is the current expected credit loss (CECL) reserve?
done: Who is the company's auditor?


,question,old answer,old pass,new answer,new source,new pass,new problems,new quote
0,What was the net interest margin this quarter?,Not found in the provided excerpts.,True,"QUOTE: Net interest margin for the three months ended June 30, 2026 was approximately $8.6 million. SOURCE: [fixed-0047]",[fixed-0047],False,"format, quote not in source","Net interest margin for the three months ended June 30, 2026 was approximately $8.6 million."
1,Which loans are on non-accrual status?,"As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $287.3 million. As of December 31, 2025, the Company had four loans held for investment on non-accrual status with a Carrying Value of $308.1 million. Additionally, as of June 30, 2026, the subordinated C-Note was on non-accrual status. Not found in the provided excerpts.",False,"QUOTE: As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $287.3 million. As of December 31, 2025, the Company had four loans held for investment on non-accrual status with a Carrying Value of $308.1 million. SOURCE: [fixed-0063]",[fixed-0063],False,format,"As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $287.3 million. As of December 31, 2025, the Company had four loans held for investment on non-accrual status with a Carrying Value of $308.1 million."
2,What dividend per share was declared for the quarter?,"The dividend per share declared for the quarter was $0.15. This information is found in the excerpts [fixed-0161], [fixed-0178], and [fixed-0271].",True,QUOTE: The Company’s board of directors declared a regular cash dividend of $0.15 per common share for the third quarter of 2026. SOURCE: [fixed-0161],[fixed-0161],False,format,The Company’s board of directors declared a regular cash dividend of $0.15 per common share for the third quarter of 2026.
3,"What was the total carrying value of loans held for investment at June 30, 2026?","The total carrying value of loans held for investment at June 30, 2026, was $1,748,835,000. This information is found in the excerpt [fixed-0281].",False,"The total carrying value of loans held for investment at June 30, 2026, was $1,748,835.",[fixed-0281],False,quote not in source,"As of June 30, 2026, the Company’s loans held for investment portfolio | $1,748,835"
4,How large is the current expected credit loss (CECL) reserve?,"The current expected credit loss (CECL) reserve as of June 30, 2026, is $139.1 million. This information is found in the excerpt [fixed-0071].",True,The current expected credit loss (CECL) reserve is $139.1 million.,[fixed-0071],True,none,"As of June 30, 2026, the Company’s CECL Reserve for its loans held for investment portfolio is $139.1 million or 723 basis points of the Company’s total loans held for investment commitment balance of $1.9 billion and is bifurcated between the CECL Reserve (contra-asset) related to outstanding balances on loans held for investment of $137.8 million and a liability for unfunded commitments of $1.3 million."
5,Who is the company's auditor?,Not found in the provided excerpts.,True,Not found in the provided excerpts.,none,True,said not found,none


In [8]:
# The same results as plain text (easier to read and to share than the table). No model calls.
for row in comparison.to_dict("records"):
    print(f"Q: {row['question']}")
    print(f"  OLD ({'pass' if row['old pass'] else 'check'}): {row['old answer']}")
    print(f"  NEW ({'pass' if row['new pass'] else 'check'}; problems: {row['new problems']}): "
          f"{row['new answer']}  {row['new source']}")
    print(f"  NEW QUOTE: {row['new quote']}\n")

Q: What was the net interest margin this quarter?
  OLD (pass): Not found in the provided excerpts.
  NEW (check; problems: format, quote not in source): QUOTE: Net interest margin for the three months ended June 30, 2026 was approximately $8.6 million.
SOURCE: [fixed-0047]  [fixed-0047]
  NEW QUOTE: Net interest margin for the three months ended June 30, 2026 was approximately $8.6 million.

Q: Which loans are on non-accrual status?
  OLD (check): As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $287.3 million. As of December 31, 2025, the Company had four loans held for investment on non-accrual status with a Carrying Value of $308.1 million. Additionally, as of June 30, 2026, the subordinated C-Note was on non-accrual status. Not found in the provided excerpts.
  NEW (check; problems: format): QUOTE: As of June 30, 2026, the Company had three loans held for investment on non-accrual status with a Carrying Value of $2